In [1]:
from qutip import Qobj
from gulps.core import GateInvariants
import numpy as np
import numpy as np
from qutip.random_objects import rand_unitary
from qutip.core.metrics import average_gate_fidelity
from tqdm import trange

# Encoded Universality under Restricted Connectivity

## Motivation

Encoded universality results (Bacon, Kempe, Lidar, Whaley) study whether restricted
*interaction Hamiltonians* (e.g., only Heisenberg exchange) can achieve universality
by encoding logical qubits into subspaces of physical qubits. In those works, the
connectivity graph is typically fully connected.

We study the **complementary** problem: interaction Hamiltonians are unrestricted
(arbitrary two-qubit unitaries), but the **connectivity graph** is restricted.

**Observation:** If there exists a path between two nodes in the connectivity graph,
then arbitrary SWAP operations along that path make them effectively fully connected.
Therefore, the nontrivial case reduces to **disconnected subgraphs** — pairs of qubits
that have no path between them whatsoever.

## Setup

We study the simplest nontrivial instance:

- **Physical qubits:** 3 qubits labeled {0, 1, 2}
- **Connectivity graph:** a single edge (1, 2). No direct gate between (0,1) or (0,2).
- **Logical partition:** qubits (0, 1) encode 1 logical qubit via isometry $V: \mathbb{C}^2 \to \mathbb{C}^4$. Qubit 2 is an ancilla.
- **Available operations:**
  - Arbitrary single-qubit gates on q0, q1, q2
  - Arbitrary two-qubit gates on (1, 2) only
- **Encoding:** $V$ is prepared once (using the (0,1) connection), then the (0,1) connection is removed.

Note that q0 is **completely disconnected** from (q1, q2) at the physical level.
The only link between q0 and the rest is through the encoding $V$.

## Question

**Does there exist an encoding $V$ such that, using only local gates and (1,2) interactions,
one can implement a universal gate set on the logical qubit?**

## Formulation

Let $W = V \otimes |0\rangle_{q2}$ be the full encoding into the 3-qubit space.
Since the available gates generate $SU(2)_{q0} \times SU(4)_{(1,2)}$, and multiple
rounds with the same encoding collapse (because $V^\dagger V = I$), the set of
achievable logical operations is:

$$\mathcal{G} = \{ W^\dagger (U_0 \otimes U_{12}) W \;:\; U_0 \in SU(2),\; U_{12} \in SU(4) \}$$

Equivalently, using the reduction $M = \langle 0|_{q2} \, U_{12} \, |0\rangle_{q2}$
(the 2x2 top-left block of $U_{12}$):

$$\mathcal{G} = \{ V^\dagger (U_0 \otimes M) V \;:\; U_0 \in SU(2),\; M \in B_1(\mathbb{C}^{2\times 2}) \}$$

where $B_1$ is the operator unit ball. The question is whether $\mathcal{G} \supseteq SU(2)$.

In [2]:
import numpy as np


def random_isometry():
    """Random isometry V: C^2 -> C^4 (Haar-random on the Stiefel manifold).

    Encodes 1 logical qubit into qubits (0,1).
    """
    A = np.random.randn(4, 2) + 1j * np.random.randn(4, 2)
    V, _ = np.linalg.qr(A)
    return V


def logical_gate(V: np.ndarray, U_phys: np.ndarray) -> np.ndarray:
    """Compute U_logical = W† (I x U_phys) W, where W = V x |0>.

    The full encoding is W = V x |0>_ancilla : C^2 -> C^8,
    embedding the logical qubit into the 3-qubit space with
    ancilla initialized to |0>.

    Args:
        V: (4, 2) isometry encoding logical qubit into qubits (0,1).
        U_phys: (4, 4) unitary acting on qubits (1,2).

    Returns:
        U_logical: (2, 2) matrix on the logical qubit.
    """
    I2 = np.eye(2)
    ket0 = np.array([[1], [0]])
    # W = V x |0> : (8, 2) — full encoding into 3-qubit space
    W = np.kron(V, ket0)
    # I_q0 x U_phys : (8, 8) — U_phys on qubits (1,2)
    U_full = np.kron(I2, U_phys)
    return W.conj().T @ U_full @ W

In [3]:
U_phys = rand_unitary(4).full()
V = random_isometry()
U_logical = logical_gate(V, U_phys)

# check how close to unitary
unitarity_error = np.linalg.norm(U_logical.conj().T @ U_logical - np.eye(2))
print(f"U_logical:\n{U_logical}\n")
print(f"||U†U - I|| = {unitarity_error:.6f}")
print(f"Singular values: {np.linalg.svd(U_logical, compute_uv=False)}")

U_logical:
[[-0.13019748+0.21349528j  0.09779701-0.16414547j]
 [-0.01139883+0.26098225j -0.07279496+0.1823744j ]]

||U†U - I|| = 1.269513
Singular values: [0.36847533 0.26470122]


In [5]:
n_samples = 1000
errors = []

for _ in trange(n_samples):
    V = random_isometry()
    best_error = np.inf
    for _ in range(n_samples):
        U_phys = rand_unitary(4).full()
        U_log = logical_gate(V, U_phys)
        err = np.linalg.norm(U_log.conj().T @ U_log - np.eye(2))
        if err < best_error:
            best_error = err
    errors.append(best_error)

errors = np.array(errors)
print(f"Unitarity error ||U†U - I|| over {n_samples} random (V, U_phys) pairs:")
print(f"  mean:   {errors.mean():.4f}")
print(f"  std:    {errors.std():.4f}")
print(f"  min:    {errors.min():.4f}")
print(f"  max:    {errors.max():.4f}")
print(f"  exact unitary (err < 1e-10): {(errors < 1e-10).sum()}/{n_samples}")

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [02:55<00:00,  5.69it/s]

Unitarity error ||U†U - I|| over 1000 random (V, U_phys) pairs:
  mean:   0.3326
  std:    0.0706
  min:    0.1084
  max:    0.5782
  exact unitary (err < 1e-10): 0/1000
